# Lab 20: XAI and Deployment

**Module 20** | Read `notes/20-shap-lime-xai-deployment.pdf` before running this notebook.

Train a fraud classifier, explain predictions with SHAP and LIME, and expose a Flask /predict endpoint.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Explainable AI and Model Deployment

Machine learning models used for fraud detection must be **interpretable** and **deployable**.
This lab covers three layers:

1. **Train** a scikit-learn Random Forest on the credit-fraud sample (V1 to V28 features).
2. **Explain** predictions with **SHAP** (tree attributions) and **LIME** (local linear approximation).
3. **Serve** the model through a minimal **Flask** REST API at `POST /predict`.

The Flask app is tested with Flask's **test client** so the notebook does not block waiting for a server process.


### Train the fraud classifier

We load the credit-fraud sample, stratify the train/test split, fit a Random Forest, and print confusion-matrix counts plus a classification report.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

parquet_path = ROOT / "datasets" / "credit_fraud_sample.parquet"
csv_path = ROOT / "datasets" / "credit_fraud_sample.csv"
if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    print(f"Loaded {parquet_path.name}: {len(df):,} rows")
else:
    df = pd.read_csv(csv_path)
    print(f"Loaded {csv_path.name}: {len(df):,} rows")

feature_cols = [f"V{i}" for i in range(1, 29)]
X = df[feature_cols].values
y = df["Class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print()
print("Confusion matrix counts (rows=actual, cols=predicted):")
print(f"              pred 0    pred 1")
print(f"  actual 0    {cm[0, 0]:6d}    {cm[0, 1]:6d}")
print(f"  actual 1    {cm[1, 0]:6d}    {cm[1, 1]:6d}")
print()
print(classification_report(y_test, y_pred, target_names=["legit", "fraud"]))

model_path = ROOT / "models" / "credit_fraud_rf.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"model": clf, "feature_cols": feature_cols}, model_path)
print(f"Saved model bundle to {model_path}")


### SHAP: global tree attributions

`TreeExplainer` computes exact Shapley values for tree ensembles.
We explain one test row and list the top features by absolute SHAP magnitude for the fraud class.


In [ ]:
import shap

explainer = shap.TreeExplainer(clf)
explain_row = 0
shap_values = explainer.shap_values(X_test[explain_row : explain_row + 1])

if isinstance(shap_values, list):
    fraud_shap = np.array(shap_values[1][0])
    print(f"SHAP values for test row {explain_row} (fraud class, positive = pushes toward fraud)")
else:
    fraud_shap = np.array(shap_values[0])
    print(f"SHAP values for test row {explain_row}")

true_label = int(y_test[explain_row])
fraud_prob = clf.predict_proba(X_test[explain_row : explain_row + 1])[0][1]
print(f"True label: {true_label} ({'fraud' if true_label == 1 else 'legit'})")
print(f"Predicted prob(fraud): {fraud_prob:.4f}")
print()
print("Top 10 features by |SHAP|:")
ranked = sorted(
    zip(feature_cols, fraud_shap),
    key=lambda x: abs(x[1]),
    reverse=True,
)
for rank, (feat, val) in enumerate(ranked[:10], start=1):
    direction = "toward fraud" if val > 0 else "toward legit"
    print(f"  {rank:2d}. {feat}: {val:+.5f}  ({direction})")


### LIME: local interpretable explanation

LIME fits a sparse linear model around a single prediction.
We explain the same test row and print the weighted feature list LIME returns.


In [ ]:
import lime.lime_tabular

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train,
    feature_names=feature_cols,
    class_names=["legit", "fraud"],
    mode="classification",
    random_state=42,
)

lime_row = explain_row
lime_exp = lime_explainer.explain_instance(
    X_test[lime_row],
    clf.predict_proba,
    num_features=10,
)

print(f"LIME explanation for test row {lime_row}")
print(f"True label: {int(y_test[lime_row])}, predicted prob(fraud): {fraud_prob:.4f}")
print()
print("LIME feature list (as returned by as_list()):")
for feat, weight in lime_exp.as_list():
    print(f"  {feat}: {weight:+.4f}")


### Flask inference API

The app exposes `GET /health` and `POST /predict`.
We exercise both endpoints with Flask's test client and pretty-print JSON responses.


In [ ]:
import json

from flask import Flask, jsonify, request

app = Flask(__name__)
_bundle = joblib.load(model_path)
_model = _bundle["model"]
_feature_cols = _bundle["feature_cols"]


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "features": len(_feature_cols)})


@app.route("/predict", methods=["POST"])
def predict():
    payload = request.get_json(force=True, silent=True) or {}
    rows = payload.get("instances")
    if not rows:
        return jsonify({"error": "Provide JSON: {'instances': [[V1,..,V28], ...]}"}), 400

    X_in = np.asarray(rows, dtype=float)
    if X_in.ndim == 1:
        X_in = X_in.reshape(1, -1)
    if X_in.shape[1] != len(_feature_cols):
        return jsonify({
            "error": f"Expected {len(_feature_cols)} features per row, got {X_in.shape[1]}"
        }), 400

    probs = _model.predict_proba(X_in)
    preds = _model.predict(X_in)
    return jsonify({
        "predictions": preds.astype(int).tolist(),
        "probabilities": probs.tolist(),
        "feature_cols": _feature_cols,
    })


def pprint_response(label: str, response) -> None:
    print(f"{label} (HTTP {response.status_code}):")
    try:
        body = response.get_json()
        print(json.dumps(body, indent=2))
    except Exception:
        print(response.data.decode())
    print()


with app.test_client() as client:
    health_resp = client.get("/health")
    pprint_response("GET /health", health_resp)

    test_rows = X_test[:3].tolist()
    pred_resp = client.post("/predict", json={"instances": test_rows})
    pprint_response("POST /predict (3 test rows)", pred_resp)

    bad_resp = client.post("/predict", json={"instances": [[1.0, 2.0]]})
    pprint_response("POST /predict (invalid feature count)", bad_resp)


### Prediction summary on held-out test rows

Quick table of true labels, predicted classes, and fraud probabilities for the first few test transactions.


In [ ]:
probs_all = clf.predict_proba(X_test)
preds_all = clf.predict(X_test)

print("=" * 58)
print("TEST ROW PREDICTION SUMMARY (first 8 rows)")
print("=" * 58)
print(f"{'Row':>4} | {'True':>4} | {'Pred':>4} | {'P(fraud)':>10} | {'Match':>5}")
print("-" * 58)
for i in range(min(8, len(X_test))):
    true_l = int(y_test[i])
    pred_l = int(preds_all[i])
    p_fraud = probs_all[i][1]
    match = "yes" if true_l == pred_l else "no"
    print(f"{i:>4} | {true_l:>4} | {pred_l:>4} | {p_fraud:>10.4f} | {match:>5}")

correct = int((preds_all == y_test).sum())
print("-" * 58)
print(f"Overall test accuracy: {correct}/{len(y_test)} = {correct / len(y_test):.4f}")


## Summary

| Component | Purpose |
|-----------|---------|
| Random Forest | Strong baseline on tabular fraud data |
| SHAP TreeExplainer | Exact Shapley values for tree models (top features for one row) |
| LIME | Human-readable local explanation for a single transaction |
| Flask `/predict` | Production-style HTTP inference (tested in-process here) |

**Next steps:** containerize the Flask app (Docker), add input validation (pydantic), and log prediction latency plus explanation metadata to MLflow for audit trails.
